# 02 – Drug response prediction v4

### Purpose
This notebook repeats the **full 5-omics baseline pipeline** (`02_model_development_multiomics.ipynb`) but adds **fold-safe batch correction** on continuous layers before each model fit. Use it to test whether removing source/batch effects improves drug-response prediction vs the uncorrected baseline.

## Integration strategies
- **Single-omics** — one model per layer
- **Early fusion** — all layers concatenated
- **Signal fusion** — expression + methylation + proteomics (if available)
- **Lineage-augmented** — fusion + OncotreeLineage dummies
- **Simple late fusion** — average of single-omics predictions
- **Learned late fusion** — RidgeCV / RidgeSelectK + non-negative stacking


## 1. Environment setup

In [3]:
from pathlib import Path
import json
import os
import time
import warnings

PROJECT_DIR   = Path.cwd()
MPLCONFIG_DIR = PROJECT_DIR / "data" / "model_results_v4" / ".matplotlib"
MPLCONFIG_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIG_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from scipy.stats import pearsonr, spearmanr
from sklearn.cross_decomposition import PLSRegression
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import ElasticNetCV, LinearRegression, RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except Exception as exc:
    XGBRegressor = None
    XGBOOST_AVAILABLE = False
    print(f"XGBoost unavailable: {exc}")

try:
    from lightgbm import LGBMRegressor
    LIGHTGBM_AVAILABLE = True
except Exception as exc:
    LGBMRegressor = None
    LIGHTGBM_AVAILABLE = False
    print(f"LightGBM unavailable: {exc}")

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


### 1.1 Layer configuration and paths

In [4]:
# Layer flags 
LAYER_CONFIG = {
    "expr": True,
    "mut":  True,
    "cnv":  True,
    "meth": False,   # set True if methylation parquet is available
    "prot": False,   # set True if proteomics parquet is available
}

ACTIVE_LAYERS        = [k for k, v in LAYER_CONFIG.items() if v]
ACTIVE_SIGNAL_LAYERS = [l for l in ["expr", "meth", "prot"] if LAYER_CONFIG.get(l)]

print(f"Active layers       : {ACTIVE_LAYERS}")
print(f"Signal fusion layers: {ACTIVE_SIGNAL_LAYERS}")

_LAYER_FILE_NAMES = {
    "expr": "expression", "mut": "mutations", "cnv": "cnv",
    "meth": "methylation", "prot": "proteomics",
}

def _find_processed_dir():
    candidates = []
    if not LAYER_CONFIG.get("prot") and not LAYER_CONFIG.get("meth"):
        candidates = [PROJECT_DIR / "data" / "processed_no_prot_no_methyl",
                      PROJECT_DIR / "data" / "processed"]
    elif not LAYER_CONFIG.get("prot"):
        candidates = [PROJECT_DIR / "data" / "processed_no_prot",
                      PROJECT_DIR / "data" / "processed"]
    else:
        candidates = [PROJECT_DIR / "data" / "processed"]

    required = [_LAYER_FILE_NAMES[l] for l in ACTIVE_LAYERS] + ["drug_response"]
    for cand in candidates:
        if not cand.exists():
            continue
        missing = [f for f in required if not (cand / f"{f}.parquet").exists()]
        if not missing:
            return cand
        print(f"  {cand.name}: missing {missing}")
    raise FileNotFoundError("No processed data directory found. Run notebook 01 first.")

PROCESSED_DIR = _find_processed_dir()
RESULTS_DIR   = PROJECT_DIR / "data" / "model_results_v4"
FIGURE_DIR    = RESULTS_DIR / "figures"
MODEL_DIR     = RESULTS_DIR / "fitted_models"
for d in [RESULTS_DIR, FIGURE_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"\nPROCESSED_DIR : {PROCESSED_DIR}")
print(f"RESULTS_DIR   : {RESULTS_DIR}")


Active layers       : ['expr', 'mut', 'cnv']
Signal fusion layers: ['expr']

PROCESSED_DIR : C:\Users\pikil\Multi-Omics-Mechanism-Modelling-AAC\data\processed_no_prot_no_methyl
RESULTS_DIR   : C:\Users\pikil\Multi-Omics-Mechanism-Modelling-AAC\data\model_results_v4


### 1.2 Model hyperparameters

In [ ]:
RUN_FULL_MODELING            = True    # False → quick mode (12 drugs)
N_ANALYSIS_COMPOUNDS         = 12
N_SPLITS                     = 5
MIN_DRUG_COVERAGE            = 250
MIN_AAC_STD                  = 0.04
TOP_N_IMPORTANCE_FEATURES    = 50
QUICK_MAX_FEATURES_PER_LAYER = 1000   # used only when RUN_FULL_MODELING=False
SELECT_K_DEFAULT             = 1000   # fallback k for RidgeSelectK
PLS_N_COMPONENTS             = 20     # PLS latent-component cap for full-run modeling

# Load per-strategy k from tuning results (02_ridge_selectk_k_tuning.ipynb)
best_k_path = RESULTS_DIR / "ridge_selectk_best_k.json"
if best_k_path.exists():
    with open(best_k_path) as f:
        best_k_payload = json.load(f)
    BEST_K_BY_STRATEGY = best_k_payload.get(
        "recommended_k_by_strategy", best_k_payload
    )
    print(f"Loaded BEST_K_BY_STRATEGY: {BEST_K_BY_STRATEGY}")
else:
    BEST_K_BY_STRATEGY = {}
    print(f"ridge_selectk_best_k.json not found — using default k={SELECT_K_DEFAULT} for all strategies")

if RUN_FULL_MODELING:
    ELASTICNET_ALPHAS    = np.logspace(-3, 0.5, 12)
    ELASTICNET_L1_RATIOS = [0.2, 0.5, 0.8]
    RIDGE_ALPHAS         = np.logspace(-2, 4, 20)
else:
    ELASTICNET_ALPHAS    = np.logspace(-3, 0.5, 5)
    ELASTICNET_L1_RATIOS = [0.5]
    RIDGE_ALPHAS         = np.logspace(-1, 3, 8)

XGBOOST_PARAMS = {
    "n_estimators": 300, "max_depth": 3, "learning_rate": 0.03,
    "subsample": 0.8, "colsample_bytree": 0.5, "reg_lambda": 5,
    "reg_alpha": 1, "objective": "reg:squarederror",
    "random_state": RANDOM_STATE, "n_jobs": -1,
    "tree_method": "hist", "verbosity": 0,
}
if not RUN_FULL_MODELING:
    XGBOOST_PARAMS.update({"n_estimators": 120, "max_depth": 2, "colsample_bytree": 0.7})

LIGHTGBM_PARAMS = {
    "n_estimators": 300, "learning_rate": 0.03, "num_leaves": 15,
    "max_depth": -1, "min_child_samples": 15, "subsample": 0.8,
    "colsample_bytree": 0.6, "reg_lambda": 5, "reg_alpha": 1,
    "objective": "regression", "random_state": RANDOM_STATE,
    "n_jobs": -1, "verbose": -1,
}
if not RUN_FULL_MODELING:
    LIGHTGBM_PARAMS.update({"n_estimators": 160, "num_leaves": 9, "colsample_bytree": 0.7})

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
print(f"RUN_FULL_MODELING: {RUN_FULL_MODELING}")
print(f"XGBoost: {XGBOOST_AVAILABLE}  LightGBM: {LIGHTGBM_AVAILABLE}")


## 2. Load and align processed matrices

In [6]:
# Load each active omics layer from its parquet file
omics_raw = {}
for key in ACTIVE_LAYERS:
    fname = _LAYER_FILE_NAMES[key]
    p = PROCESSED_DIR / f"{fname}.parquet"
    if not p.exists():
        raise FileNotFoundError(f"Layer '{key}' active but file missing: {p}")
    df = pd.read_parquet(p)
    if "ModelID" in df.columns:
        df = df.set_index("ModelID")
    # Remove leftover index columns from to_csv/read_csv without index=False
    df = df.drop(columns=[c for c in df.columns if str(c).startswith("Unnamed")],
                 errors="ignore")
    omics_raw[key] = df
    print(f"  Loaded {key:<5}: {df.shape}")

drug_response_raw = pd.read_parquet(PROCESSED_DIR / "drug_response.parquet")
if "ModelID" in drug_response_raw.columns:
    drug_response_raw = drug_response_raw.set_index("ModelID")

# Load cell metadata (OncotreeLineage, etc.); optional — may not exist
meta_path = PROCESSED_DIR / "cell_metadata.parquet"
cell_metadata = pd.read_parquet(meta_path) if meta_path.exists() else None
if cell_metadata is not None and "ModelID" in cell_metadata.columns:
    cell_metadata = cell_metadata.set_index("ModelID")

# Enrich metadata with SourceType from raw DepMap Model.csv (used as batch proxy)
model_source_path = PROJECT_DIR / "data" / "raw" / "Model.csv"
if model_source_path.exists():
    model_source = (
        pd.read_csv(model_source_path, usecols=["ModelID", "SourceType"])
        .dropna(subset=["ModelID"]).drop_duplicates("ModelID").set_index("ModelID")
    )
    cell_metadata = (cell_metadata.join(model_source, how="left")
                     if cell_metadata is not None else model_source)
    print(f"SourceType loaded: {model_source['SourceType'].nunique()} unique sources")

# Find cell lines present in ALL active omics layers AND drug response simultaneously;
# any line missing even one layer is excluded to guarantee a complete feature matrix
common_ids = sorted(
    set.intersection(*(set(df.index) for df in omics_raw.values()),
                     set(drug_response_raw.index))
)

# Align all omics layers to the shared cell-line index;
# fill any residual NaNs with 0 and cast to float32 for memory efficiency
omics = {
    name: df.loc[common_ids].sort_index().fillna(0.0).astype(np.float32)
    for name, df in omics_raw.items()
}

# Align drug response (target matrix) to the same index
drug_response = drug_response_raw.loc[common_ids].sort_index().astype(np.float32)

# Build one-hot lineage feature block from OncotreeLineage;
# lineage is passed as explicit features rather than regressed out of omics layers (v4 design)
if cell_metadata is not None:
    cell_metadata = cell_metadata.loc[
        cell_metadata.index.intersection(common_ids)
    ].sort_index()
    lineage_features = pd.get_dummies(
        cell_metadata.reindex(common_ids)["OncotreeLineage"].fillna("Unknown"),
        prefix="lineage", dtype=np.float32,
    )
    # Prefix with "meta::" to distinguish lineage columns from omics features downstream
    lineage_features.columns = [f"meta::{col}" for col in lineage_features.columns]
    lineage_features = lineage_features.loc[drug_response.index]
else:
    lineage_features = pd.DataFrame(index=drug_response.index)

print(f"\nShared cell lines : {len(common_ids)}")
print(f"Drug response     : {drug_response.shape}")
print(f"Lineage features  : {lineage_features.shape[1]}")

# Sanity check: every omics layer must have the exact same index as drug_response
# (same cell lines, same order) — mismatch here would silently corrupt X/y alignment
for name, df in omics.items():
    assert df.index.equals(drug_response.index), f"Index mismatch: {name}"
    print(f"  {name:<5} {df.shape}")

  Loaded expr : (574, 4999)
  Loaded mut  : (574, 1000)
  Loaded cnv  : (574, 4999)
SourceType loaded: 65 unique sources

Shared cell lines : 574
Drug response     : (574, 544)
Lineage features  : 24
  expr  (574, 4999)
  mut   (574, 1000)
  cnv   (574, 4999)


## 2.1 Fold-safe batch correction helpers

ComBat-style location/scale harmonisation fitted **only on training samples** of each fold,
then applied to both train and test. Prevents leakage.
Batch labels = DepMap `SourceType` (source repository proxy).
Corrected layers: expression, methylation, proteomics 


In [7]:
from typing import Optional, Set, Tuple

_ALL_BATCH_CORRECT = {"expr", "meth", "prot"}
BATCH_CORRECT_LAYERS = _ALL_BATCH_CORRECT & set(ACTIVE_LAYERS)
print(f"Batch-corrected layers: {sorted(BATCH_CORRECT_LAYERS)}")


def _resolve_batch_labels(index: pd.Index) -> pd.Series:
    if cell_metadata is not None:
        for col in ["Batch", "batch", "BatchID", "SourceType", "Source"]:
            if col in cell_metadata.columns:
                labels = cell_metadata.reindex(index)[col].fillna("unknown").astype(str)
                if labels.nunique() > 1:
                    return labels
    # Fallback: single batch (no correction possible)
    return pd.Series("unknown", index=index)


batch_labels = _resolve_batch_labels(pd.Index(common_ids))
print("Batch label distribution:")
print(batch_labels.value_counts().head(10))
print(f"Total: {len(batch_labels)} cells, {batch_labels.nunique()} source types")


def _combat_fit(train_df: pd.DataFrame, batch: pd.Series) -> dict:
    batch = pd.Series(batch).reindex(train_df.index).fillna("unknown").astype(str)
    X = train_df.astype(np.float32).fillna(train_df.median(axis=0))
    params = {
        "grand_mean": X.mean(axis=0),
        "grand_var":  X.var(axis=0, ddof=1).replace(0, 1.0),
        "batches": {},
    }
    for b in sorted(batch.unique()):
        mask = batch == b
        if mask.sum() < 2:
            continue
        params["batches"][b] = {
            "mean": X.loc[mask].mean(axis=0),
            "var":  X.loc[mask].var(axis=0, ddof=1).replace(0, 1.0),
        }
    return params


def _combat_transform(df: pd.DataFrame, batch: pd.Series, params: dict) -> pd.DataFrame:
    batch = pd.Series(batch).reindex(df.index).fillna("unknown").astype(str)
    X = df.astype(np.float32).fillna(df.median(axis=0))
    if not params["batches"]:
        return X
    adjusted = X.copy()
    for b, stats in params["batches"].items():
        mask = batch == b
        if not mask.any():
            continue
        adjusted.loc[mask] = (
            (X.loc[mask] - stats["mean"])
            * np.sqrt(params["grand_var"] / stats["var"])
            + params["grand_mean"]
        ).astype(np.float32)
    return adjusted


def _layer_columns(X: pd.DataFrame, layer: str,
                   omics_layer: Optional[str] = None) -> list:
    prefixed = [c for c in X.columns if str(c).startswith(f"{layer}::")]
    if prefixed:
        return prefixed
    if omics_layer == layer and layer in BATCH_CORRECT_LAYERS:
        return list(X.columns)
    return []


def fold_correct_features(
    X: pd.DataFrame,
    train_ids,
    test_ids,
    omics_layer: Optional[str] = None,
    layers: Optional[Set[str]] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Fit ComBat on train_ids, transform train+test. No leakage."""
    if layers is None:
        layers = BATCH_CORRECT_LAYERS
    train_ids = pd.Index(train_ids)
    test_ids  = pd.Index(test_ids)
    fold_ids  = train_ids.union(test_ids)

    X_train = X.loc[train_ids].copy()
    X_test  = X.loc[test_ids].copy()
    batch_train = batch_labels.loc[train_ids]
    batch_all   = batch_labels.loc[fold_ids]

    for layer in layers:
        cols = _layer_columns(X, layer, omics_layer=omics_layer)
        if not cols:
            continue
        params    = _combat_fit(X_train[cols], batch_train)
        corrected = _combat_transform(X.loc[fold_ids, cols], batch_all, params)
        X_train[cols] = corrected.loc[train_ids]
        X_test[cols]  = corrected.loc[test_ids]

    return X_train, X_test


Batch-corrected layers: ['expr']
Batch label distribution:
SourceType
ATCC                   255
DSMZ                   118
HSRRB                   67
KCLB                    36
RIKEN                   32
ECACC                   32
JCRB                     6
NCI/DCTD                 5
The Broad Institute      5
Academic lab             4
Name: count, dtype: int64
Total: 574 cells, 17 source types


## 3. Select eligible drug targets

In [8]:
drug_summary = pd.DataFrame({
    "compound":  drug_response.columns,
    "coverage":  drug_response.notna().sum(axis=0).values,
    "aac_mean":  drug_response.mean(axis=0, skipna=True).values,
    "aac_std":   drug_response.std(axis=0,  skipna=True).values,
    "aac_min":   drug_response.min(axis=0,  skipna=True).values,
    "aac_max":   drug_response.max(axis=0,  skipna=True).values,
}).set_index("compound")

drug_summary["eligible"]        = (
    (drug_summary["coverage"] >= MIN_DRUG_COVERAGE) &
    (drug_summary["aac_std"]  >= MIN_AAC_STD)
)
drug_summary["selection_score"] = drug_summary["coverage"] * drug_summary["aac_std"]

eligible_compounds = (
    drug_summary.loc[drug_summary["eligible"]]
    .sort_values(["selection_score", "coverage", "aac_std"], ascending=False)
)
eligible_compounds.reset_index().to_csv(RESULTS_DIR / "eligible_compounds.csv", index=False)

print(f"Total compounds    : {drug_response.shape[1]:,}")
print(f"Eligible compounds : {len(eligible_compounds):,}")
display(eligible_compounds.head(10))


Total compounds    : 544
Eligible compounds : 426


,coverage,aac_mean,aac_std,aac_min,aac_max,eligible,selection_score
compound,,,,,,,
Vincristine,556,0.496922,0.234956,0.000000,0.987693,True,130.635700
SB-743921,522,0.584016,0.242093,0.030407,0.987152,True,126.372428
Oligomycin A,508,0.383554,0.244020,0.000000,0.981403,True,123.962258
BI-2536,537,0.516770,0.214197,0.000000,0.990110,True,115.023786
docetaxel:tanespimycin (2:1 mol/mol),539,0.582816,0.207719,0.000000,0.983458,True,111.960492
"1S,3R-RSL-3",544,0.457454,0.200701,0.013289,0.924021,True,109.181570
KX2-391,522,0.401344,0.197750,0.000000,0.921312,True,103.225509
Cytarabine,520,0.219701,0.197939,0.000000,0.973958,True,102.928300
Leptomycin B,549,0.716831,0.186601,0.000000,0.984921,True,102.443760


## 4. Build feature sets

In [9]:
SINGLE_LAYERS        = ACTIVE_LAYERS
SIGNAL_FUSION_LAYERS = ACTIVE_SIGNAL_LAYERS

LAYER_LABELS = {
    "expr": "Expression", "mut": "Mutations", "cnv": "CNV",
    "meth": "Methylation", "prot": "Proteomics",
    "multiomics": "Multi-omics",
    "multiomics_plus_lineage": "Multi-omics + lineage",
    "signal_multiomics": "Expression + meth + prot",
    "expr_plus_lineage": "Expression + lineage",
}


def select_features_by_variance(df: pd.DataFrame, k: Optional[int] = None) -> pd.DataFrame:
    if k is None or k >= df.shape[1]:
        return df.copy()
    top_cols = df.var(axis=0, ddof=1).fillna(0.0).nlargest(k).index
    return df.loc[:, top_cols].copy()


feature_cap = None if RUN_FULL_MODELING else QUICK_MAX_FEATURES_PER_LAYER
omics_for_modeling = {
    name: select_features_by_variance(df, k=feature_cap)
    for name, df in omics.items()
}

feature_sets = {name: omics_for_modeling[name] for name in SINGLE_LAYERS}

early_fusion = pd.concat(
    [omics_for_modeling[n].rename(columns=lambda c, n=n: f"{n}::{c}")
     for n in SINGLE_LAYERS],
    axis=1,
)
feature_sets["early_fusion"] = early_fusion

if len(SIGNAL_FUSION_LAYERS) >= 2:
    signal_fusion = pd.concat(
        [omics_for_modeling[n].rename(columns=lambda c, n=n: f"{n}::{c}")
         for n in SIGNAL_FUSION_LAYERS],
        axis=1,
    )
    feature_sets["signal_fusion"] = signal_fusion

if not lineage_features.empty:
    aligned_lineage = lineage_features.loc[early_fusion.index]
    feature_sets["early_fusion_lineage"] = pd.concat(
        [early_fusion, aligned_lineage], axis=1
    )
    if "signal_fusion" in feature_sets:
        feature_sets["signal_fusion_lineage"] = pd.concat(
            [feature_sets["signal_fusion"], aligned_lineage], axis=1
        )
    feature_sets["expr_lineage"] = pd.concat(
        [omics_for_modeling["expr"].rename(columns=lambda c: f"expr::{c}"),
         aligned_lineage],
        axis=1,
    )

print("Feature set shapes:")
for name, df in feature_sets.items():
    print(f"  {name:<28} {df.shape[0]:>4,} x {df.shape[1]:>6,}")


Feature set shapes:
  expr                          574 x  4,999
  mut                           574 x  1,000
  cnv                           574 x  4,999
  early_fusion                  574 x 10,998
  early_fusion_lineage          574 x 11,022
  expr_lineage                  574 x  5,023


## 5. Modeling setup

`get_model(model_family, strategy)` creates a fresh model instance.
`RidgeSelectK` uses a **per-strategy k** loaded from `ridge_selectk_best_k.json`
(produced by `02_ridge_selectk_k_tuning.ipynb`). If the file is absent, falls back to `SELECT_K_DEFAULT`.
`StandardScaler` is always inside `make_pipeline` — fitted only on training fold data.


In [ ]:
def positive_int_or_default(value, fallback: int) -> int:
    try:
        value = int(value)
    except (TypeError, ValueError):
        value = int(fallback)
    return max(1, value)


def select_k_for_strategy(strategy: str, n_features: Optional[int] = None) -> int:
    raw_k = BEST_K_BY_STRATEGY.get(strategy, SELECT_K_DEFAULT)
    k = positive_int_or_default(raw_k, SELECT_K_DEFAULT)
    return min(k, int(n_features)) if n_features is not None else k


def pls_components_for_fold(
    n_features: Optional[int] = None,
    n_samples: Optional[int] = None,
) -> int:
    limits = [PLS_N_COMPONENTS]
    if n_features is not None:
        limits.append(int(n_features))
    if n_samples is not None:
        limits.append(max(1, int(n_samples) - 1))
    return max(1, min(limits))


def get_model(
    model_family: str,
    strategy: str,
    n_features: Optional[int] = None,
    n_samples: Optional[int] = None,
):
    """Return a fresh unfitted model for the given family and strategy."""
    if model_family == "ElasticNet":
        return make_pipeline(
            StandardScaler(with_mean=True, with_std=True),
            ElasticNetCV(
                l1_ratio=ELASTICNET_L1_RATIOS, alphas=ELASTICNET_ALPHAS,
                cv=3, max_iter=10_000, random_state=RANDOM_STATE,
                n_jobs=-1, selection="random",
            ),
        )
    if model_family == "RidgeCV":
        return make_pipeline(
            StandardScaler(with_mean=True, with_std=True),
            RidgeCV(alphas=RIDGE_ALPHAS),
        )
    if model_family == "RidgeSelectK":
        k = select_k_for_strategy(strategy, n_features=n_features)
        return make_pipeline(
            StandardScaler(with_mean=True, with_std=True),
            SelectKBest(score_func=f_regression, k=k),
            RidgeCV(alphas=RIDGE_ALPHAS),
        )
    if model_family == "PLSRegression":
        n_components = pls_components_for_fold(
            n_features=n_features, n_samples=n_samples
        )
        return make_pipeline(
            StandardScaler(with_mean=True, with_std=True),
            PLSRegression(n_components=n_components, scale=False),
        )
    if model_family == "XGBoost":
        return XGBRegressor(**XGBOOST_PARAMS)
    if model_family == "LightGBM":
        return LGBMRegressor(**LIGHTGBM_PARAMS)
    raise ValueError(f"Unknown model_family: {model_family}")


MODEL_FAMILIES = ["ElasticNet", "RidgeCV", "RidgeSelectK", "PLSRegression"]
if XGBOOST_AVAILABLE:
    MODEL_FAMILIES.append("XGBoost")
if LIGHTGBM_AVAILABLE:
    MODEL_FAMILIES.append("LightGBM")

print("Model families:", ", ".join(MODEL_FAMILIES))
print(f"Per-strategy k: {BEST_K_BY_STRATEGY or 'default=' + str(SELECT_K_DEFAULT)}")
print(f"PLS components cap: {PLS_N_COMPONENTS}")


### 5.1 Scoring helper

In [11]:
def score_predictions(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ok     = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[ok], y_pred[ok]
    if len(y_true) == 0:
        return dict(pearson=np.nan, spearman=np.nan, rmse=np.nan, mae=np.nan, r2=np.nan)
    if len(y_true) < 3 or np.std(y_true) == 0 or np.std(y_pred) == 0:
        pearson = spearman = np.nan
    else:
        pearson  = float(pearsonr(y_true,  y_pred).statistic)
        spearman = float(spearmanr(y_true, y_pred).statistic)
    return {
        "pearson":  pearson,
        "spearman": spearman,
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae":  float(mean_absolute_error(y_true, y_pred)),
        "r2":   float(r2_score(y_true, y_pred)) if len(y_true) > 1 else np.nan,
    }


### 5.2 Cross-validation helper

In [12]:
# Which (model_family, strategy) combos to save models for SHAP analysis.
# Set to None to save everything (large disk usage).
SAVE_CONFIGS: Optional[Set[Tuple[str, str]]] = None  # save all


def _model_save_path(model_family: str, compound: str,
                     strategy: str, fold_id: int) -> Path:
    safe = (compound.replace(" ", "_").replace("/", "-")
                    .replace("(", "").replace(")", "")
                    .replace(":", "-"))
    return MODEL_DIR / f"{model_family}__{safe}__{strategy}__fold{fold_id}.pkl"


def run_cv_model(model_family, X, y, folds, compound, strategy, omics_layer):
    metric_records, prediction_records           = [], []
    feature_importance_records, layer_imp_records = [], []
    feature_names = X.columns.to_numpy()

    should_save = (SAVE_CONFIGS is None or
                   (model_family, strategy) in SAVE_CONFIGS)

    for fold_id, (train_ids, test_ids) in enumerate(folds, start=1):
        # Fold-safe batch correction (no-op for layers not in BATCH_CORRECT_LAYERS)
        X_train_df, X_test_df = fold_correct_features(
            X, train_ids, test_ids, omics_layer=omics_layer
        )
        X_train = X_train_df.to_numpy(dtype=np.float32, copy=False)
        X_test  = X_test_df.to_numpy(dtype=np.float32,  copy=False)
        y_train = y.loc[train_ids].to_numpy(dtype=np.float32)
        y_test  = y.loc[test_ids].to_numpy(dtype=np.float32)

        model = get_model(
            model_family, strategy,
            n_features=X_train.shape[1], n_samples=X_train.shape[0],
        )
        model.fit(X_train, y_train)
        y_pred = np.asarray(model.predict(X_test), dtype=float).ravel()

        metric_records.append({
            "compound": compound, "model_family": model_family,
            "strategy": strategy, "omics_layer": omics_layer,
            "fold": fold_id, "n_train": len(train_ids), "n_test": len(test_ids),
            **score_predictions(y_test, y_pred),
        })
        prediction_records.extend({
            "compound": compound, "model_family": model_family,
            "strategy": strategy, "omics_layer": omics_layer,
            "fold": fold_id, "ModelID": mid,
            "y_true": float(obs), "y_pred": float(pred),
        } for mid, obs, pred in zip(test_ids, y_test, y_pred))

        # Feature importances
        if model_family == "ElasticNet":
            values = np.abs(model.named_steps["elasticnetcv"].coef_)
        elif model_family == "RidgeCV":
            values = np.abs(model.named_steps["ridgecv"].coef_).ravel()
        elif model_family == "RidgeSelectK":
            selector = model.named_steps["selectkbest"]
            values   = np.zeros(len(feature_names), dtype=float)
            values[selector.get_support()] = np.abs(
                model.named_steps["ridgecv"].coef_
            ).ravel()
        elif model_family == "PLSRegression":
            coef = np.asarray(
                model.named_steps["plsregression"].coef_, dtype=float
            ).ravel()
            values = (
                np.abs(coef) if coef.size == len(feature_names)
                else np.zeros(len(feature_names))
            )
        elif model_family in {"XGBoost", "LightGBM"}:
            values = np.asarray(model.feature_importances_, dtype=float)
        else:
            values = np.zeros(len(feature_names), dtype=float)

        feat_df = pd.DataFrame({
            "feature":    feature_names.astype(str),
            "omics_layer": [str(f).split("::", 1)[0] if "::" in str(f)
                            else omics_layer for f in feature_names],
            "importance": np.asarray(values, dtype=float).ravel(),
        }).sort_values("importance", ascending=False)

        layer_df = (
            feat_df.groupby("omics_layer", as_index=False)["importance"]
            .sum().rename(columns={"importance": "raw_importance"})
        )
        total = layer_df["raw_importance"].sum()
        layer_df["importance_fraction"] = (
            layer_df["raw_importance"] / total if total > 0 else np.nan
        )

        top_feats = feat_df.head(TOP_N_IMPORTANCE_FEATURES).assign(
            compound=compound, model_family=model_family,
            strategy=strategy, omics_layer_model=omics_layer, fold=fold_id,
        )
        feature_importance_records.extend(
            top_feats[["compound","model_family","strategy",
                        "omics_layer_model","fold","feature",
                        "omics_layer","importance"]].to_dict("records")
        )
        layer_imp_records.extend(
            layer_df.assign(compound=compound, model_family=model_family,
                            strategy=strategy, omics_layer_model=omics_layer,
                            fold=fold_id)
            [["compound","model_family","strategy","omics_layer_model",
              "fold","omics_layer","raw_importance","importance_fraction"]]
            .to_dict("records")
        )

        if should_save:
            joblib.dump(model, _model_save_path(model_family, compound, strategy, fold_id))

    return (metric_records, prediction_records,
            feature_importance_records, layer_imp_records)


## 6. Cross-validated modeling

In [13]:
modeling_compounds = eligible_compounds.index.to_list()
if not RUN_FULL_MODELING:
    modeling_compounds = modeling_compounds[:N_ANALYSIS_COMPOUNDS]
print(f"Modeling {len(modeling_compounds)} compounds "
      f"({'full' if RUN_FULL_MODELING else 'quick'} mode)")

evaluation_specs = [
    (layer_name, "single_omics", layer_name) for layer_name in SINGLE_LAYERS
]
evaluation_specs.append(("early_fusion", "early_fusion", "multiomics"))
if "signal_fusion" in feature_sets:
    evaluation_specs.append(("signal_fusion", "signal_fusion", "signal_multiomics"))
if "early_fusion_lineage" in feature_sets:
    evaluation_specs.append(
        ("early_fusion_lineage", "early_fusion_lineage", "multiomics_plus_lineage")
    )
if "signal_fusion_lineage" in feature_sets:
    evaluation_specs.append(
        ("signal_fusion_lineage", "signal_fusion_lineage",
         "signal_multiomics_plus_lineage")
    )
if "expr_lineage" in feature_sets:
    evaluation_specs.append(
        ("expr_lineage", "metadata_augmented", "expr_plus_lineage")
    )

print(f"Evaluation specs ({len(evaluation_specs)}):")
for feat_key, strat, layer in evaluation_specs:
    print(f"  {strat:<28} | {layer}")


Modeling 426 compounds (full mode)
Evaluation specs (6):
  single_omics                 | expr
  single_omics                 | mut
  single_omics                 | cnv
  early_fusion                 | multiomics
  early_fusion_lineage         | multiomics_plus_lineage
  metadata_augmented           | expr_plus_lineage


### 6.1 Base model cross-validation

In [ ]:
all_metrics, all_predictions           = [], []
all_feature_importance, all_layer_imp  = [], []

started = time.time()
for compound_i, compound in enumerate(modeling_compounds, start=1):
    y         = drug_response[compound].dropna()
    valid_ids = y.index.to_numpy()
    kf        = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    folds     = [(valid_ids[tr], valid_ids[te]) for tr, te in kf.split(valid_ids)]

    print(f"[{compound_i:>3}/{len(modeling_compounds):>3}] {compound}  n={len(y)}")

    for model_family in MODEL_FAMILIES:
        family_predictions = []
        for feature_key, strategy_name, omics_layer_name in evaluation_specs:
            m, p, f, l = run_cv_model(
                model_family, feature_sets[feature_key],
                y, folds, compound, strategy_name, omics_layer_name,
            )
            all_metrics.extend(m)
            all_predictions.extend(p)
            all_feature_importance.extend(f)
            all_layer_imp.extend(l)
            if strategy_name == "single_omics":
                family_predictions.extend(p)

        # Simple late fusion (average of single-omics predictions)
        single_pred_df = pd.DataFrame(family_predictions)
        if not single_pred_df.empty:
            averaged = (
                single_pred_df
                .groupby(["fold", "ModelID"], as_index=False)
                .agg(y_true=("y_true", "first"), y_pred=("y_pred", "mean"))
            )
            _lf_layer = f"late_fusion({'_'.join(SINGLE_LAYERS)})"
            for fold_id, fold_df in averaged.groupby("fold"):
                all_predictions.extend({
                    "compound": compound, "model_family": model_family,
                    "strategy": "late_fusion", "omics_layer": _lf_layer,
                    "fold": int(fold_id), "ModelID": row.ModelID,
                    "y_true": float(row.y_true), "y_pred": float(row.y_pred),
                } for row in fold_df.itertuples())
                all_metrics.append({
                    "compound": compound, "model_family": model_family,
                    "strategy": "late_fusion", "omics_layer": _lf_layer,
                    "fold": int(fold_id), "n_train": np.nan,
                    "n_test": len(fold_df),
                    **score_predictions(fold_df["y_true"], fold_df["y_pred"]),
                })

elapsed = (time.time() - started) / 60
print(f"\nFinished {len(modeling_compounds)} compounds in {elapsed:.1f} min")


[  1/426] Vincristine  n=556


C:\Users\pikil\anaconda3\envs\pytorch\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\pikil\anaconda3\envs\pytorch\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\pikil\anaconda3\envs\pytorch\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\pikil\anaconda3\envs\pytorch\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\pikil\anaconda3\envs\pytorch\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with featur

[  2/426] SB-743921  n=522


### 6.2 Save base model outputs

In [ ]:
cv_metrics         = pd.DataFrame(all_metrics)
cv_predictions     = pd.DataFrame(all_predictions)
feature_importance = pd.DataFrame(all_feature_importance)
layer_importance   = pd.DataFrame(all_layer_imp)

cv_metrics.to_csv(RESULTS_DIR / "cv_metrics.csv", index=False)
cv_predictions.to_csv(RESULTS_DIR / "cv_predictions.csv", index=False)
if not feature_importance.empty:
    feature_importance.to_csv(
        RESULTS_DIR / "feature_importance_all_models.csv", index=False
    )
if not layer_importance.empty:
    layer_importance.to_csv(
        RESULTS_DIR / "omics_layer_importance.csv", index=False
    )

print("Saved base model outputs:")
for path in sorted(RESULTS_DIR.glob("*.csv")):
    print(f"  {path.name:<38} {path.stat().st_size / 1024:>9.1f} KB")
display(cv_metrics.head(3))


## 7. Learned late fusion (non-negative stacking)

In [ ]:
learned_metrics, learned_predictions, learned_weights = [], [], []

stacking_specs = [
    ("RidgeCV",      "RidgeCV+PositiveStack"),
    ("RidgeSelectK", "RidgeSelectK+PositiveStack"),
]

started_llf = time.time()
for base_family, model_label in stacking_specs:
    for compound_i, compound in enumerate(modeling_compounds, start=1):
        y         = drug_response[compound].dropna()
        valid_ids = y.index.to_numpy()
        kf        = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
        folds     = [(valid_ids[tr], valid_ids[te]) for tr, te in kf.split(valid_ids)]

        print(f"[{model_label} {compound_i:>3}/{len(modeling_compounds):>3}] "
              f"{compound}  n={len(y)}")

        for fold_id, (train_ids, test_ids) in enumerate(folds, start=1):
            z_train = pd.DataFrame(index=train_ids, columns=SINGLE_LAYERS, dtype=float)
            z_test  = pd.DataFrame(index=test_ids,  columns=SINGLE_LAYERS, dtype=float)
            inner_cv = KFold(n_splits=3, shuffle=True,
                             random_state=RANDOM_STATE + fold_id)

            for layer_name in SINGLE_LAYERS:
                X = feature_sets[layer_name]
                for inner_tr_idx, inner_val_idx in inner_cv.split(train_ids):
                    inner_tr_ids  = train_ids[inner_tr_idx]
                    inner_val_ids = train_ids[inner_val_idx]
                    base_model    = get_model(base_family, "single_omics")
                    X_itr, X_ival = fold_correct_features(
                        X, inner_tr_ids, inner_val_ids, omics_layer=layer_name
                    )
                    base_model.fit(
                        X_itr.to_numpy(dtype=np.float32, copy=False),
                        y.loc[inner_tr_ids].to_numpy(dtype=np.float32),
                    )
                    z_train.loc[inner_val_ids, layer_name] = np.asarray(
                        base_model.predict(
                            X_ival.to_numpy(dtype=np.float32, copy=False)
                        ), dtype=float
                    ).ravel()

                final_base = get_model(base_family, "single_omics")
                X_tr, X_te = fold_correct_features(
                    X, train_ids, test_ids, omics_layer=layer_name
                )
                final_base.fit(
                    X_tr.to_numpy(dtype=np.float32, copy=False),
                    y.loc[train_ids].to_numpy(dtype=np.float32),
                )
                z_test[layer_name] = np.asarray(
                    final_base.predict(X_te.to_numpy(dtype=np.float32, copy=False)),
                    dtype=float,
                ).ravel()

            meta_model = LinearRegression(positive=True)
            meta_model.fit(
                z_train.to_numpy(dtype=float),
                y.loc[train_ids].to_numpy(dtype=float),
            )
            y_pred = np.asarray(
                meta_model.predict(z_test.to_numpy(dtype=float)), dtype=float
            ).ravel()
            y_test = y.loc[test_ids].to_numpy(dtype=float)

            learned_metrics.append({
                "compound": compound, "model_family": model_label,
                "strategy": "learned_late_fusion", "omics_layer": "multiomics",
                "fold": fold_id, "n_train": len(train_ids), "n_test": len(test_ids),
                **score_predictions(y_test, y_pred),
            })
            learned_predictions.extend({
                "compound": compound, "model_family": model_label,
                "strategy": "learned_late_fusion", "omics_layer": "multiomics",
                "fold": fold_id, "ModelID": mid,
                "y_true": float(obs), "y_pred": float(pred),
            } for mid, obs, pred in zip(test_ids, y_test, y_pred))

            coef     = np.asarray(meta_model.coef_, dtype=float).ravel()
            coef_sum = np.abs(coef).sum()
            for layer_name, weight in zip(SINGLE_LAYERS, coef):
                learned_weights.append({
                    "compound": compound, "model_family": model_label,
                    "strategy": "learned_late_fusion", "omics_layer": layer_name,
                    "fold": fold_id, "weight": float(weight),
                    "weight_fraction": (float(abs(weight) / coef_sum)
                                        if coef_sum > 0 else np.nan),
                })

print(f"Learned late fusion done in "
      f"{(time.time() - started_llf) / 60:.1f} min")


### 7.1 Save learned fusion outputs

In [ ]:
learned_metrics_df     = pd.DataFrame(learned_metrics)
learned_predictions_df = pd.DataFrame(learned_predictions)
learned_weights_df     = pd.DataFrame(learned_weights)

cv_metrics     = pd.concat([cv_metrics,     learned_metrics_df],     ignore_index=True)
cv_predictions = pd.concat([cv_predictions, learned_predictions_df], ignore_index=True)

learned_weights_df.to_csv(RESULTS_DIR / "learned_late_fusion_weights.csv", index=False)
cv_metrics.to_csv(RESULTS_DIR / "cv_metrics.csv", index=False)
cv_predictions.to_csv(RESULTS_DIR / "cv_predictions.csv", index=False)

if not learned_weights_df.empty:
    weight_summary = (
        learned_weights_df
        .groupby(["model_family", "omics_layer"], as_index=False)["weight_fraction"]
        .mean()
        .sort_values(["model_family", "weight_fraction"], ascending=[True, False])
    )
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(data=weight_summary, x="weight_fraction", y="omics_layer",
                hue="model_family", ax=ax)
    ax.set_xlabel("Mean learned weight fraction")
    ax.set_ylabel("Omics layer")
    ax.set_title("Learned late-fusion omics layer weights")
    plt.tight_layout()
    fig.savefig(FIGURE_DIR / "learned_late_fusion_weights.png",
                dpi=180, bbox_inches="tight")
    plt.show()

if not learned_metrics_df.empty:
    display(
        learned_metrics_df
        .groupby(["model_family", "strategy", "omics_layer"])
        [["pearson", "spearman", "rmse", "mae", "r2"]]
        .mean()
    )


## 8. Within-lineage prediction analysis

Evaluates prediction quality **within** each cancer lineage (pooled across folds).
A substantial drop in within-lineage vs overall Pearson r indicates the model is
exploiting tissue-of-origin structure rather than drug-specific biology.


In [ ]:
def compute_within_lineage_metrics(
    cv_preds: pd.DataFrame,
    meta: pd.DataFrame,
    min_samples: int = 15,
) -> pd.DataFrame:
    if meta is None:
        print("No cell metadata — skipping within-lineage analysis")
        return pd.DataFrame()

    if isinstance(meta.index.name, str) and meta.index.name == "ModelID":
        lineage_map = meta["OncotreeLineage"].to_dict()
    elif "ModelID" in meta.columns:
        lineage_map = meta.set_index("ModelID")["OncotreeLineage"].to_dict()
    else:
        lineage_map = {}

    preds = cv_preds.copy()
    preds["lineage"] = preds["ModelID"].map(lineage_map).fillna("Unknown")

    records = []
    for (compound, model_family, strategy, lineage), grp in preds.groupby(
        ["compound", "model_family", "strategy", "lineage"]
    ):
        if len(grp) < min_samples:
            continue
        metrics = score_predictions(grp["y_true"].values, grp["y_pred"].values)
        records.append({
            "compound": compound, "model_family": model_family,
            "strategy": strategy, "lineage": lineage,
            "n_samples": len(grp), **metrics,
        })
    return pd.DataFrame(records)


within_lineage_df = compute_within_lineage_metrics(
    cv_predictions, cell_metadata, min_samples=15
)

if not within_lineage_df.empty:
    within_lineage_df.to_csv(
        RESULTS_DIR / "within_lineage_metrics.csv", index=False
    )
    print(f"Within-lineage metrics: {within_lineage_df.shape}")

    overall_mean = cv_metrics.groupby(["model_family", "strategy"])["pearson"].mean()
    within_mean  = within_lineage_df.groupby(["model_family", "strategy"])["pearson"].mean()
    comparison = pd.DataFrame({
        "overall_pearson":        overall_mean,
        "within_lineage_pearson": within_mean,
    }).dropna()
    comparison["drop"] = comparison["overall_pearson"] - comparison["within_lineage_pearson"]
    comparison.to_csv(RESULTS_DIR / "within_lineage_comparison.csv")

    print("\nOverall vs within-lineage Pearson r (top 20):")
    display(comparison.sort_values("within_lineage_pearson", ascending=False).head(20))

    # Per-lineage boxplot for best model
    best_combo = cv_metrics.groupby(["model_family","strategy"])["pearson"].mean().idxmax()
    best_family, best_strategy = best_combo
    wl_sub = within_lineage_df[
        (within_lineage_df["model_family"] == best_family) &
        (within_lineage_df["strategy"]     == best_strategy)
    ]
    if not wl_sub.empty:
        top_lin  = wl_sub.groupby("lineage")["pearson"].median().nlargest(15).index
        wl_plot  = wl_sub[wl_sub["lineage"].isin(top_lin)]
        fig, ax  = plt.subplots(figsize=(12, 6))
        sns.boxplot(data=wl_plot, x="pearson", y="lineage",
                    order=top_lin, ax=ax, color="steelblue")
        ax.set_xlabel("Within-lineage Pearson r")
        ax.set_ylabel("Cancer lineage")
        ax.set_title(f"Within-lineage: {best_family} / {best_strategy}")
        ax.axvline(0, color="gray", linestyle="--", linewidth=0.8)
        plt.tight_layout()
        fig.savefig(FIGURE_DIR / "within_lineage_pearson.png",
                    dpi=180, bbox_inches="tight")
        plt.show()
else:
    print("No within-lineage results computed.")


## 9. Overall results summary

In [ ]:
def add_expression_ridge_reference(metrics: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:

    mask = (
        (metrics["model_family"] == "RidgeCV") &
        (metrics["strategy"] == "single_omics") &
        (metrics["omics_layer"] == "expr")
    )
    reference = metrics.loc[mask].copy()
    if reference.empty:
        return metrics.copy(), reference
    reference["model_family"] = "ExpressionOnlyRidge"
    reference["strategy"] = "reference_baseline"
    reference["omics_layer"] = "expr"
    return pd.concat([metrics, reference], ignore_index=True), reference


cv_metrics_for_summary, expression_ridge_reference = add_expression_ridge_reference(cv_metrics)

core_summary = (
    cv_metrics.groupby(["model_family", "strategy"])
    .agg(
        pearson_mean=("pearson",  "mean"),
        pearson_std =("pearson",  "std"),
        spearman_mean=("spearman","mean"),
        rmse_mean   =("rmse",     "mean"),
        r2_mean     =("r2",       "mean"),
        n_folds     =("pearson",  "count"),
    )
    .sort_values("pearson_mean", ascending=False)
)
summary = (
    cv_metrics_for_summary.groupby(["model_family", "strategy"])
    .agg(
        pearson_mean=("pearson",  "mean"),
        pearson_std =("pearson",  "std"),
        spearman_mean=("spearman","mean"),
        rmse_mean   =("rmse",     "mean"),
        r2_mean     =("r2",       "mean"),
        n_folds     =("pearson",  "count"),
    )
    .sort_values("pearson_mean", ascending=False)
)
core_summary.to_csv(RESULTS_DIR / "cv_metrics_summary.csv")
summary.to_csv(RESULTS_DIR / "cv_metrics_summary_with_reference_baselines.csv")

if not expression_ridge_reference.empty:
    reference_summary = (
        expression_ridge_reference
        .groupby(["model_family", "strategy", "omics_layer"])
        .agg(
            pearson_mean=("pearson",  "mean"),
            pearson_std =("pearson",  "std"),
            spearman_mean=("spearman","mean"),
            rmse_mean   =("rmse",     "mean"),
            r2_mean     =("r2",       "mean"),
            n_compounds =("compound", "nunique"),
            n_folds     =("pearson",  "count"),
        )
        .sort_values("pearson_mean", ascending=False)
    )
    reference_summary.to_csv(RESULTS_DIR / "reference_baselines.csv")
    print("Named reference baseline:")
    display(reference_summary.round(4))

print("Top 20 model-strategy combinations by mean Pearson r, including named references:")
display(summary.head(20))


In [ ]:
top_strategies = (
    cv_metrics.groupby("strategy")["pearson"]
    .mean().sort_values(ascending=False).head(8).index
)
plot_data = cv_metrics[cv_metrics["strategy"].isin(top_strategies)].copy()

fig, ax = plt.subplots(figsize=(14, 6))
sns.boxplot(data=plot_data, x="strategy", y="pearson",
            hue="model_family", order=top_strategies, ax=ax)
ax.set_xlabel("Strategy")
ax.set_ylabel("Pearson r (per compound × fold)")
ax.set_title("Drug response prediction performance (v4 — batch-corrected, all drugs)")
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.tick_params(axis="x", rotation=30)
plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
fig.savefig(FIGURE_DIR / "cv_pearson_by_strategy.png",
            dpi=180, bbox_inches="tight")
plt.show()


## 10. SHAP handover summary

In [ ]:
config_summary = (
    cv_metrics
    .groupby(["model_family", "strategy", "omics_layer"], as_index=False)
    .agg(
        mean_pearson =("pearson",  "mean"),
        mean_spearman=("spearman", "mean"),
        mean_r2      =("r2",       "mean"),
        mean_rmse    =("rmse",     "mean"),
        n_folds      =("pearson",  "count"),
    )
    .sort_values(["mean_pearson", "mean_spearman"], ascending=False)
    .reset_index(drop=True)
)
config_summary["rank"] = config_summary.index + 1
top10 = config_summary.head(10)
top10.to_csv(RESULTS_DIR / "top10_model_strategy_summary.csv", index=False)

print("Top 10 model × strategy × omics_layer")
display(top10[["rank","model_family","strategy","omics_layer",
               "mean_pearson","mean_r2","mean_rmse","n_folds"]].round(4))

best        = top10.iloc[0]
best_mask   = (
    (cv_metrics["model_family"] == best["model_family"]) &
    (cv_metrics["strategy"]     == best["strategy"])     &
    (cv_metrics["omics_layer"]  == best["omics_layer"])
)
per_drug = (
    cv_metrics.loc[best_mask]
    .groupby("compound", as_index=False)
    .agg(mean_pearson=("pearson","mean"), mean_r2=("r2","mean"),
         n_folds=("pearson","count"))
    .sort_values("mean_pearson", ascending=False)
)
shap_drugs = per_drug.head(3)["compound"].tolist()

print(f"\nBest config:")
print(f"  model_family : {best['model_family']}")
print(f"  strategy     : {best['strategy']}")
print(f"  omics_layer  : {best['omics_layer']}")
print(f"  mean_pearson : {best['mean_pearson']:.4f}")
print(f"  SHAP drugs   : {', '.join(shap_drugs)}")
print(f"  Models saved : {MODEL_DIR}")
print(f"\nSHAP explainer hints:")
print("  RidgeCV / ElasticNet / RidgeSelectK → shap.LinearExplainer")
print("  PLSRegression                       → coefficient or permutation analysis")
print("  XGBoost / LightGBM                 → shap.TreeExplainer")

handover = pd.DataFrame([{
    "model_family":               best["model_family"],
    "strategy":                   best["strategy"],
    "omics_layer":                best["omics_layer"],
    "mean_pearson":               best["mean_pearson"],
    "mean_r2":                    best["mean_r2"],
    "active_layers":              ", ".join(ACTIVE_LAYERS),
    "batch_corrected_layers":     ", ".join(sorted(BATCH_CORRECT_LAYERS)),
    "recommended_shap_compounds": "; ".join(shap_drugs),
    "notebook":                   "02_model_development_v4.ipynb",
    "processed_dir":              str(PROCESSED_DIR),
    "results_dir":                str(RESULTS_DIR),
    "models_dir":                 str(MODEL_DIR),
}])
handover.to_csv(RESULTS_DIR / "shap_handover_recommendation.csv", index=False)
print(f"\nSaved SHAP handover: {RESULTS_DIR / 'shap_handover_recommendation.csv'}")
display(handover)

print("\nPer-drug Pearson for best config:")
display(per_drug.round(4))
